# Description

This code generate a co-citation graph of the corpus. The output are in PDF and PGF formats.
Each reference appearing in the corpus are processed: the Levenstein distance is computed to compare reference to one another.

# Prerequisites

- matplotlib
- pandas
- openpyxl
- Levenstein
- [grobid](https://grobid.readthedocs.io/en/latest/Grobid-docker/) to generate the tei.xml files of each reference.


Use the following to install the packages:

```pip install -e ".[citation_network]"```


## A note on 'grobid'

Use the lightweight version to generate the tei.xml file, using the following command:

```
docker run -t --rm -p 8070:8070 lfoppiano/grobid:0.8.2
```

The server is available on your machine at localhost, port 8070: http://localhost:8070/

In the tab 'TEI', select the 'Process all References' in 'Process to call' and check the option 'Consolidate citation'; select the PDF file of the paper you want to process and then clic on 'Submit'. The resulting file is then saved.

# Input

You need a `filename` in excel format in the `data` folder.
The `RAW` sheet must contain at least the following columns:
- 'Title'
- 'BiblCitation' containing a string with \cite{xxx} in LaTeX format
- 'TeiFile'

# Output 

- Two files named `output_file`.ext, with the following extensions:
    - PGF
    - PDF


In [ ]:
import pandas as pd

import reference_man
import random
import numpy as np



In [ ]:
filename = '../data/refs.xlsx'
sheet_name = 'RAW'
output_file = "cocitation_graph"
seed = 42
random.seed(seed)
np.random.seed(seed)
levenstein_threshold = 0.75
num_leafs = 20

In [ ]:
corpus = pd.read_excel(filename, sheet_name=sheet_name)

In [ ]:
refs = {}

In [ ]:
corpus_titles = list(corpus['Title'])

In [ ]:
refs_dict = {}

In [ ]:
for id,item in corpus.iterrows():
    pub = reference_man.Publication.from_xlsx(item)
    pub_title = pub.get_title()
    refs[pub_title] = set()
    for reference in pub.get_references():
        ref_title = reference.get_title()
        if ref_title is not None:
            if ref_title not in refs:
                refs[ref_title] = set()

In [ ]:
import Levenshtein


In [ ]:
for id,item in corpus.iterrows():
    pub = reference_man.Publication.from_xlsx(item)
    pub_title = pub.get_title()
    for reference in pub.get_references():
        ref_title = reference.get_title()

In [ ]:
for id,item in corpus.iterrows():
    pub = reference_man.Publication.from_xlsx(item)
    pub_title = pub.get_title()

    print("-----",pub_title)
    print("nb references=",len(pub.get_references()))
    refs_dict[pub_title] = pub.get_references()
    for reference in pub.get_references():
        ref_title = reference.get_title()
        if ref_title is not None:
            match = None
            for t in refs.keys():
                if Levenshtein.ratio(t,ref_title) > levenstein_threshold: 
                    match = t
                    break

            if match:
                refs[pub_title].add(match)

In [ ]:
use_pfg = True

if use_pfg:
    import matplotlib as mpl

    mpl.use("pgf")
    mpl.rcParams.update({
        "pgf.texsystem": "pdflatex",   
        "font.family": "serif",
        "text.usetex": True,
        "pgf.rcfonts": False,    
    })


In [ ]:
from matplotlib import pyplot as plt
import networkx as nx

from itertools import combinations

G = nx.Graph()

corpus_titles = set(refs.keys())

for id,item in corpus.iterrows():
    doc = item['Title']
    references = refs[doc]
    label_name = item["BiblCitation"]
    G.add_node(doc, type="corpus",label=label_name)

    for ref in references:
        if ref != doc:
            G.add_edge(doc, ref, weight=1)

labels = nx.get_node_attributes(G, "label")


print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())



In [ ]:
for corpus_node in [n for n, d in G.nodes(data=True) if d.get("type") == "corpus"]:
    neighbors = list(G.neighbors(corpus_node))

    leaf_neighbors = [n for n in neighbors if G.degree[n] == 1 and G.nodes[n].get("type") != "corpus"]

    keep = set(leaf_neighbors[:num_leafs])

    remove = [n for n in leaf_neighbors if n not in keep]
    G.remove_nodes_from(remove)

print("Final graph :", G.number_of_nodes(), "nodes and ", G.number_of_edges(), " edges")

In [ ]:
low_degree_nodes = [n for n, d in G.degree() if d <= 1]
G.remove_nodes_from(low_degree_nodes)


In [ ]:
from adjustText import adjust_text

fig, ax = plt.subplots(figsize=(14, 10))

pos = nx.spring_layout(G, seed=10, k=0.55, iterations=100)

red_edges = [
    (u, v) for u, v in G.edges()
    if G.nodes[u].get("type") == "corpus" and G.nodes[v].get("type") == "corpus"
]
gray_edges = [
    (u, v) for u, v in G.edges()
    if (u, v) not in red_edges
]

nx.draw_networkx_edges(G, pos, edgelist=gray_edges, edge_color="gray",
                       alpha=0.4, width=0.6, arrowstyle='-', arrows=True,
                       connectionstyle="arc3,rad=0.1", ax=ax)

nx.draw_networkx_edges(G, pos, edgelist=red_edges, edge_color="red",
                       width=2, alpha=0.8, arrowstyle='-', arrows=True,
                       connectionstyle="arc3,rad=0.1", ax=ax)

corpus_nodes = [n for n, d in G.nodes(data=True) if d.get("type") == "corpus"]
ref_nodes    = [n for n, d in G.nodes(data=True) if d.get("type") != "corpus"]

nx.draw_networkx_nodes(G, pos, nodelist=ref_nodes, node_size=1, ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=corpus_nodes, node_color="red", node_size=10, ax=ax)

texts = []
for node, label in labels.items():
    x, y = pos[node]
    t = ax.text(x, y, label, fontsize=12, va="bottom", ha="left")
    texts.append(t)

adjust_text(
    texts,
    ax=ax,
    arrowprops=dict(arrowstyle="-", color="gray", lw=0.4)
)

In [ ]:

plt.savefig(output_file+".pgf")  
plt.savefig(output_file+".pdf")  
